<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 02. K-Means: Agrupando por Cercanía a un Centro
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 10
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/10%20-%20Clustering/Para%20Dummies/02_KMeans_Clustering_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del módulo 02 de Clustering. El cuaderno principal habló de "minimizar la suma de cuadrados intra-cluster" y de "`k-means++`" — aquí vamos a entender la misma idea con una analogía sencilla y el mismo dataset de clientes.

Al terminar podrás explicar, con tus propias palabras:
1. Cómo funciona el algoritmo K-Means, paso a paso.
2. Cómo aplicarlo en scikit-learn sobre `mall_customers.csv`.
3. Por qué la posición inicial de los "centros" importa, y qué hace `k-means++`.
4. En qué tipo de datos K-Means funciona bien, y en cuáles falla.

---
## 1. La analogía de los puntos de encuentro 🚩

Imagina que organizas una excursión con 200 personas repartidas por un parque enorme, y quieres dividirlas en 5 grupos, cada uno con su propio guía parado en un punto de encuentro. El proceso podría ser así:

1. Plantas 5 banderas (los "puntos de encuentro") en posiciones más o menos al azar.
2. Le pides a cada persona que camine hacia la bandera **más cercana** a ella.
3. Una vez que todos están agrupados, cada guía se mueve al **centro exacto** de su grupo (el promedio de dónde quedó parada su gente).
4. Como los guías se movieron, algunas personas ahora están más cerca de otra bandera — así que vuelves a pedirles que caminen hacia la bandera más cercana.
5. Repites esto hasta que ya nadie cambia de grupo: las banderas dejaron de moverse.

Eso es exactamente **K-Means**: "banderas" = **centroides**, y "caminar hacia la más cercana" = **asignar cada punto al centroide más cercano**. La letra "K" es, sencillamente, el número de banderas (grupos) que tú decides usar de antemano.

---
## Configuración del entorno de trabajo 🛠️

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df_mall = pd.read_csv('../data/mall_customers.csv')
print("Dataset cargado:", df_mall.shape)
df_mall.head()

---
## 2. Preparando el terreno: ingreso y puntaje de gasto 🛍️

Vamos a agrupar a los 200 clientes usando dos columnas: `Annual_Income_k` (ingreso anual, en miles) y `Spending_Score` (qué tanto gasta, de 1 a 100). Primero, veamos los puntos sin agrupar.

In [ ]:
X = df_mall[['Annual_Income_k', 'Spending_Score']].values

plt.figure(figsize=(6.5, 5))
plt.scatter(X[:, 0], X[:, 1], color="#475569", s=30, alpha=0.75)
plt.title("Clientes sin agrupar")
plt.xlabel("Ingreso anual (miles de USD)")
plt.ylabel("Puntaje de gasto (1-100)")
plt.show()

A simple vista ya se distinguen aproximadamente 5 "nubes" de puntos. Antes de aplicar K-Means, **estandarizamos** las variables con `StandardScaler` (como aprendimos en el cuaderno 01): K-Means usa distancia Euclidiana por dentro, así que cualquier diferencia de escala entre columnas podría distorsionar los grupos.

In [ ]:
escalador = StandardScaler()
X_escalado = escalador.fit_transform(X)

print("Media después de escalar (debe ser ~0):", X_escalado.mean(axis=0).round(4))
print("Dispersión después de escalar (debe ser ~1):", X_escalado.std(axis=0).round(4))

---
## 3. Plantando las 5 banderas: `KMeans` en scikit-learn ⚡

Le pedimos a `KMeans` que forme 5 grupos (`n_clusters=5`). Los parámetros más importantes para entender por ahora:

* `n_clusters`: cuántas "banderas" (grupos) queremos — este número lo decidimos nosotros.
* `n_init`: cuántas veces repite todo el proceso con banderas iniciales distintas, quedándose con el mejor resultado (más adelante veremos por qué esto importa).
* `random_state`: una semilla para que el resultado sea reproducible (el mismo cada vez que corras el código).

In [ ]:
modelo_kmeans = KMeans(n_clusters=5, init='k-means++', n_init=10, random_state=42)
grupo_asignado = modelo_kmeans.fit_predict(X_escalado)

print("Grupo de los primeros 10 clientes:", grupo_asignado[:10])
print("¿Qué tan compactos quedaron los grupos? (inercia):", round(modelo_kmeans.inertia_, 2))

### 🤔 ¿Qué acaba de pasar?

- `fit_predict` hace todo el proceso de la analogía de las banderas: coloca 5 centroides iniciales, asigna cada cliente al más cercano, mueve los centroides al centro de su grupo, y repite hasta que nadie cambia de grupo.
- El resultado, `grupo_asignado`, es un número del 0 al 4 para cada uno de los 200 clientes — ese número es "a qué bandera terminó caminando" cada persona.
- La `inertia_` mide qué tan "apretados" quedaron los grupos alrededor de su centroide (mientras más bajo, más compactos) — no necesitamos memorizar la fórmula exacta por ahora, solo saber que sirve para comparar qué tan bien quedó el agrupamiento.

In [ ]:
# Traemos los centroides de vuelta a la escala original (ingreso y puntaje reales) para graficarlos
centros_originales = escalador.inverse_transform(modelo_kmeans.cluster_centers_)

plt.figure(figsize=(6.5, 5))
plt.scatter(X[:, 0], X[:, 1], c=grupo_asignado, cmap='viridis', s=30, alpha=0.85)
plt.scatter(
    centros_originales[:, 0], centros_originales[:, 1],
    c='red', marker='X', s=220, edgecolor='black', label='Centro de cada grupo'
)
plt.title("K-Means (5 grupos): Segmentación de Clientes")
plt.xlabel("Ingreso anual (miles de USD)")
plt.ylabel("Puntaje de gasto (1-100)")
plt.legend()
plt.show()

### 🤔 ¿Qué acaba de pasar?

- Como entrenamos el modelo con datos escalados, sus centroides también están "en la escala escalada" — con `inverse_transform` los devolvemos a las unidades originales (miles de USD y puntaje de 1-100) para poder graficarlos junto a los datos reales.
- El resultado son 5 grupos con sentido de negocio real, por ejemplo: clientes de ingreso bajo y gasto bajo (conservadores), ingreso bajo y gasto alto (gastan por encima de su capacidad), ingreso medio y gasto medio (el grupo "promedio"), ingreso alto y gasto bajo (poco compromiso, buen objetivo de marketing), e ingreso alto y gasto alto (los más rentables, candidatos a un programa VIP).
- Nota que **todos** los clientes quedaron asignados a algún grupo — K-Means, a diferencia de otros algoritmos que veremos más adelante en el módulo, no tiene el concepto de "cliente que no pertenece a ningún grupo".

---
## 4. ¿Por qué importa dónde caen las banderas al principio? 🎯

Volviendo a la analogía: si una de las 5 banderas iniciales cae, por mala suerte, muy pegada a otra, puede terminar "robándole" muy pocas personas y formando un grupo diminuto y sin sentido, mientras las demás banderas se reparten mal al resto del parque. A esto se le llama quedar atrapado en una **mala solución** (un "óptimo local").

Para evitarlo, scikit-learn usa por defecto una estrategia llamada **`k-means++`**: en lugar de tirar las 5 banderas totalmente al azar, va colocándolas de forma que tiendan a quedar **bien repartidas** desde el principio (prefiere zonas alejadas de las banderas ya puestas). Además, el parámetro `n_init` repite todo el experimento varias veces con puntos de partida distintos, y se queda con el mejor resultado — como intentar la excursión varias veces y quedarte con el reparto que dejó a todos más cómodos.

In [ ]:
# Comparamos qué tan compactos quedan los grupos con distintas estrategias de inicio
resultados = []
for estrategia in ['random', 'k-means++']:
    for repeticiones in [1, 10]:
        km = KMeans(n_clusters=5, init=estrategia, n_init=repeticiones, random_state=42)
        km.fit(X_escalado)
        resultados.append({'estrategia_inicio': estrategia, 'n_init': repeticiones, 'inercia': round(km.inertia_, 3)})

pd.DataFrame(resultados)

### 🤔 ¿Qué acaba de pasar?

- Probamos las dos estrategias de inicio (`random` vs. `k-means++`) combinadas con pocas o muchas repeticiones (`n_init`).
- En general, `k-means++` con `n_init` alto (como 10) tiende a dar la menor inercia — es decir, los grupos más compactos y "bien acomodados" — porque reduce el riesgo de que una mala tirada de banderas iniciales arruine el resultado.
- Por eso `k-means++` es la opción por defecto en scikit-learn: casi siempre es la mejor opción sin que tengas que pensarlo.

---
## 5. Cuando K-Means no es la herramienta correcta ⚠️

K-Means asume que los grupos son más o menos **redondos y de tamaño parecido**, porque siempre mide distancia en línea recta al centro. Eso significa que falla en formas "raras", como dos medias lunas entrelazadas.

In [ ]:
from sklearn.datasets import make_moons

X_lunas, _ = make_moons(n_samples=300, noise=0.06, random_state=42)
kmeans_lunas = KMeans(n_clusters=2, n_init=10, random_state=42)
grupo_lunas = kmeans_lunas.fit_predict(X_lunas)

plt.figure(figsize=(6.5, 5))
plt.scatter(X_lunas[:, 0], X_lunas[:, 1], c=grupo_lunas, cmap='viridis', s=25, alpha=0.85)
plt.scatter(
    kmeans_lunas.cluster_centers_[:, 0], kmeans_lunas.cluster_centers_[:, 1],
    c='red', marker='X', s=200, edgecolor='black', label='Centro de cada grupo'
)
plt.title("K-Means falla con formas no redondas")
plt.legend()
plt.show()

### 🤔 ¿Qué acaba de pasar?

- Generamos dos "medias lunas" entrelazadas — para el ojo humano, dos grupos clarísimos y bien separados.
- K-Means, en cambio, las corta con una línea más o menos recta por la mitad, porque solo entiende "distancia al centro más cercano", no la forma real de cada grupo.
- Esta limitación es justo lo que motiva los siguientes algoritmos del módulo, que pueden reconocer grupos de formas más libres.

**En resumen:** usa K-Means cuando esperas grupos razonablemente compactos y redondeados (como los clientes del centro comercial); considera otras alternativas cuando sospeches formas irregulares, tamaños muy distintos entre grupos, o presencia fuerte de valores atípicos.

---
## 6. Resumen relámpago ⚡

| Idea | En una frase |
|---|---|
| K-Means | Algoritmo que agrupa datos asignándolos al "centro" (centroide) más cercano, y recalculando ese centro una y otra vez. |
| `n_clusters` (K) | El número de grupos que tú decides de antemano. |
| Centroide | El punto promedio de todos los datos de un grupo — se recalcula en cada ronda. |
| `k-means++` | Estrategia por defecto para colocar los centroides iniciales bien repartidos, evitando malos arranques. |
| `n_init` | Cuántas veces se repite todo el proceso con puntos de partida distintos, quedándose con el mejor resultado. |
| Limitación clave | K-Means asume grupos redondeados y de tamaño parecido — falla con formas irregulares como medias lunas o espirales. |

➡️ **Siguiente paso:** el próximo cuaderno de este módulo, **Clustering Jerárquico y Dendrogramas**, presenta una alternativa a K-Means que no obliga a fijar el número de grupos de antemano y permite explorar la estructura de los datos a distintos niveles de detalle.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>
